# MorphIQ SDK for Python — Standard-five Greeks

Value a synthetic option and inspect delta, gamma, theta, vega and rho.
Run setup once, adjust inputs, then click **Calculate — 1 request**. Setup and
control changes make no API requests. BSM uses European exercise; BS1993 uses
American exercise. The displayed snapshot records the submitted inputs and CU.
Greek units are unscaled API units: time derivatives per year, vega/rho per unit
volatility/rate. Unavailable values retain their reason; they are not zero.
Set `MORPHIQ_API_KEY` before starting, or enter it at the masked prompt on execution.

[Back to start](../start-here.ipynb)


In [ ]:
import os
from getpass import getpass

import ipywidgets as widgets
from IPython.display import display

from morphiq_sdk import GreeksStandardRequest, MorphIQ, MorphIQError

model = widgets.Dropdown(
    options=[
        ("BSM (European)", "black_scholes_merton"),
        ("BS1993 (American)", "bjerksund_stensland_1993"),
    ],
    description="Model",
)
option_type = widgets.Dropdown(options=["call", "put"], description="Option")
spot = widgets.FloatText(value=100, description="Spot")
strike = widgets.FloatText(value=100, description="Strike")
expiry = widgets.FloatText(value=1, description="Years")
volatility = widgets.FloatText(value=0.2, description="Volatility")
rate = widgets.FloatText(value=0.05, description="Rate")
dividend_yield = widgets.FloatText(value=0, description="Yield")
calculate_button = widgets.Button(
    description="Calculate — 1 request", layout=widgets.Layout(width="260px")
)
output = widgets.Output()
last_result = None
last_request = None


def calculate(_):
    global last_result, last_request
    last_result = None
    calculate_button.disabled = True
    with output:
        output.clear_output(wait=True)
        try:
            request = GreeksStandardRequest(
                exercise_style="european" if model.value == "black_scholes_merton" else "american",
                pricing_model=model.value,
                option_type=option_type.value,
                spot=spot.value,
                strike=strike.value,
                time_to_expiry_years=expiry.value,
                risk_free_rate=rate.value,
                dividend_yield=dividend_yield.value,
                volatility=volatility.value,
            )
            last_request = request.to_wire()
            api_key = os.environ.get("MORPHIQ_API_KEY") or getpass("MorphIQ API key: ")
            with MorphIQ(api_key=api_key) as client:
                last_result = client.greeks_standard(request)
            print("Inputs:", last_request)
            print("Price:", last_result.result.price)
            for name in ("delta", "gamma", "theta", "vega", "rho"):
                value = getattr(last_result.result.greeks, name)
                reason = last_result.result.greeks.unavailable.get(name)
                print(name, value if value is not None else reason)
            print("Model:", last_result.result.model_version)
            print("Request ID:", last_result.metadata.request_id)
            charged = last_result.metadata.fcu_charged
            print("CU charged:", charged if charged is not None else "not reported")
        except (MorphIQError, ValueError) as error:
            print(error)
            metadata = getattr(error, "metadata", None)
            if metadata is not None:
                print("Error evidence:", metadata.model_dump(exclude_unset=True))
        finally:
            calculate_button.disabled = False


calculate_button.on_click(calculate)
display(
    widgets.VBox(
        [
            model,
            option_type,
            spot,
            strike,
            expiry,
            volatility,
            rate,
            dividend_yield,
            calculate_button,
            output,
        ]
    )
)